In [ ]:
from matplotlib import pyplot as plt
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import seaborn as sns
import pandas as pd

sns.set_theme(style="darkgrid")
 
Rotsedata = []
Pandata = []

In [ ]:
with open('/users/cdcook/VSP/datafiles/PSdata/xtetrans_1b_Exp4Final.csv', newline='') as w:
    data = list(csv.reader(w))
    data.pop(0)

In [ ]:
def lineAt(x, d):
    return d
def oneDFit(a,b,x):
    return a*x+b
def panDat(data, band, kronCut, kronDist, kronBand, bitFlagsTF, bitFlags, max):
    print(kronCut, kronDist, kronBand, bitFlags, max)
    counter = 0
    RotseData = []
    PanData = []
    binFlags = []
    if(bitFlagsTF):
        for n in range(max):
            binFlags.append(np.binary_repr(int(data[n][5]), width=8))
            if(kronCut):
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0 and binFlags[n] == bitFlags and float(data[n][kronBand]) - float(data[n][kronBand + 1]) < kronDist and float(data[n][kronBand]) - float(data[n][kronBand + 1]) > -1*kronDist:
                    x = float(data[n][4])  
                    y = float(data[n][band])
                    RotseData.append(x)
                    PanData.append(y)
                    counter = counter + 1
            else:
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0 and binFlags[n] == bitFlags: 
                    x = float(data[n][4])  
                    y = float(data[n][band])
                    RotseData.append(x)
                    PanData.append(y)
                    counter = counter + 1
    else:
        for n in range(max):
            binFlags.append(np.binary_repr(int(data[n][5]), width=8))
            if(kronCut):
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0 and float(data[n][kronBand]) - float(data[n][kronBand + 1]) < kronDist and float(data[n][kronBand]) - float(data[n][kronBand + 1]) > -1*kronDist:
                    x = float(data[n][4])  
                    y = float(data[n][band])
                    RotseData.append(x)
                    PanData.append(y)
                    counter = counter + 1
            else:
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0: 
                    x = float(data[n][4])  
                    y = float(data[n][band])
                    RotseData.append(x)
                    PanData.append(y)
                    counter = counter + 1          
    return RotseData, PanData, counter

In [ ]:
band = 14
kronCut = False
kronDist = 0.5
kronBand = 6
bitFlagsTF = False
bitFlags = "00000000"
max = 8090

if (band == 6):
    bandname = 'g'
if (band == 8):
    bandname = 'r'
if (band == 10):
    bandname = 'i'
if (band == 12):
    bandname = 'z'
if (band == 14):
    bandname = 'y'

if (kronBand == 6):
    kronName = 'g'
if (kronBand == 8):
    kronName = 'r'
if (kronBand == 10):
    kronName = 'i'
if (kronBand == 12):
    kronName = 'z'
if (kronBand == 14):
    kronName = 'y'

In [ ]:
Rotsedata, Pandata, counter = panDat(data, band, kronCut, kronDist, kronBand, bitFlagsTF, bitFlags, max)

In [ ]:
params = np.polyfit(Rotsedata, Pandata, 1, full=False, cov=True)        
difList = []
gaxCount = 0
n = 0
for i in Pandata:
    difList.append(i - (oneDFit(params[0][0], params[0][1], Rotsedata[n])))
    if(i > oneDFit(params[0][0], params[0][1], Rotsedata[n]) + 0.8):
        gaxCount = gaxCount + 1
    n = n + 1

x = np.linspace(8, 20, 10000)    

line0 = []
for i in x:
    line0.append(lineAt(i, 0))
    
line0dot8 = []
for i in x:
    line0dot8.append(lineAt(i, 0.8))

In [ ]:
plt.figure(figsize=(18,9))
print("slope: ", params[0][0])
print("AB offset: ", params[0][1])
print("Total Count: ", counter)
print("Above red line: ", gaxCount)   
ax = sns.scatterplot(x=Rotsedata, y=Pandata)
plt.plot(x, vsp.oneDFit(params[0][0], params[0][1], x), 'k--')

ax.text(9,18,"slope: %4.4f" % params[0][0], fontsize=20)
ax.text(9,17,"AB offset: %4.4f" % params[0][1], fontsize=20)
ax.text(9,16,"Total Count: %s" % counter, fontsize=20)

#plt.plot(x, vsp.oneDFit(params[0][0], params[0][1], x) + 0.8, 'r--')
plt.xlabel('ROTSE Mag', fontsize=12)
plt.ylabel('Pan Mag - ' + bandname, fontsize=12)
#plt.title(bandname + 'Mag zero point ' + kronName + "kron cut +-" + str(kronDist) + " w/ no raised e-flags", fontsize = 20)
plt.title(bandname + 'Mag zero point', fontsize = 20)
plt.show()